In [ ]:
from datetime import date, timedelta
import requests
import pandas as pd
import time


API_KEY = "5mr300973e6o"
headers = {"X-eBirdApiToken": API_KEY}
results = [] 

def get_bird_data(Region, Date):
    URL = f"https://api.ebird.org/v2/data/obs/{Region}/historic/{Date}"

    parameters = {
        "includeProvisional": "true", #Take unverified sightings into account to get more data
        "hotspot": "false"
    }

    #Request start
    print(f" start request for {Region} on the {Date}...")      
    response = requests.get(URL, headers=headers, params=parameters)
    #check the connection
    if response.status_code == 200:
        data = response.json()
        total_bird_count = sum(item.get("howMany", 0) for item in data)
        #append wanted data, which will later be the columns of the csv file
        results.append({
            "Date": Date,
            "Region": Region,
            "Total_Amount": total_bird_count
        })
    
    else:
        print(f"Error message: {response.status_code}")
        print(response.text)

#wanted time interval
start_date = date(2021,1,1)
end_date = date(2025,12,31)

total_days = (end_date - start_date).days
# § LLM help to use timedelta
for i in range(total_days + 1):
    current_date = start_date + timedelta(days=i)

    formatted_date = current_date.strftime("%Y/%m/%d")
    #calling the function
    get_bird_data("DE-HH", formatted_date)

    time.sleep(0.2)

#Setting up DataFrame and saving results into csv file
df = pd.DataFrame(results)
df.to_csv("bird_observations_hamburg_2021-2025.csv", index=False)
print("Data saved as: bird_observations_hamburg_2020-2025.csv")
print(df.head())        #Test showing the first 5 rows

print("-------DONE-------")

In [ ]:
from datetime import datetime
import requests
import pandas as pd
import time


API_KEY = "c1a858d369eed336eb46904f1c13db80"
#Lat and Long for Hamburg, neccessary to work with the API
LAT = 53.551086
LONG = 9.993682
CITY = "Hamburg"

air_pollution_results = []

def get_air_pollution_data(start_time, end_time):

    URL = "http://api.openweathermap.org/data/2.5/air_pollution/history"

    parameters = {
        "lat": LAT,
        "lon": LONG,
        "start": start_time,
        "end": end_time,
        "appid": API_KEY
    }

    response = requests.get(URL, params=parameters)

    if response.status_code == 200:
        data = response.json()
        # § LLM help to transform date
        for entry in data.get("list", []):                                    
            dt = datetime.fromtimestamp(entry["dt"]).strftime("%Y-%m-%d")
            # § LLM help  
            air_values = entry["components"]

            print(f"Start Request for {CITY} on the {dt}")
            #Columns of the csv
            air_pollution_results.append({
                "Date": dt,
                "PM2.5": air_values.get("pm2_5"),
                "PM10": air_values.get("pm10")
            })

    else:
        print(f"Error message: {response.status_code}")
        print(response.text)

years = [2021,2022,2023,2024,2025]

for year in years:
    print(f"collecting air Data for {year}...")
    start_dt = datetime(year, 1, 1, 0, 0)       
    end_dt = datetime(year, 12, 31, 23, 59)
    # § LLM help to convert into unix timestamp 
    start_time = int(start_dt.timestamp())
    end_time = int(end_dt.timestamp())
    #calling the function
    get_air_pollution_data(start_time, end_time)

    time.sleep(1)

df = pd.DataFrame(air_pollution_results)
# § LLM help to calculate the mean and for reset_index()
df_hamburg_daily = df.groupby("Date").mean().reset_index()

df_hamburg_daily.to_csv("Hamburg_air_pollution_average_2021-2025.csv", index=False)

print(df_hamburg_daily.head())
print("------DONE------")

In [ ]:
#merging the two csv's
import pandas as pd

df_bird_counts = pd.read_csv("bird_observations_hamburg_2021-2025.csv") 
df_pollution = pd.read_csv("hamburg_air_pollution_average_2021-2025.csv")

# § LLM help to convert dates in both csv's to uniform date format
df_bird_counts["Date"] = pd.to_datetime(df_bird_counts["Date"])
df_pollution["Date"] = pd.to_datetime(df_pollution["Date"])

#merging the two files on Date
df_master = pd.merge(df_bird_counts, df_pollution, on="Date", how="inner")

df_master.to_csv("hamburg_birdCounts_pollution_2021-2025.csv")
print("csv merged!")